## Exercise 1

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 14: Modell-Deployment
# Niveau: Anfänger
# Aufgabe 1 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import os

np.random.seed(42)
tf.random.set_seed(42)

print("=== Modell exportieren und laden: 3 Formate ===\n")

# ── 1. Daten laden und Modell trainieren ──────────────────
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0
x_train = x_train.reshape(-1, 784)
x_test  = x_test.reshape(-1, 784)

model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(784,)),
    layers.Dense(64,  activation='relu'),
    layers.Dense(10,  activation='softmax')
], name='mnist_model')

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=5, batch_size=256, verbose=1)
_, orig_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nOriginalmodell Test-Accuracy: {orig_acc:.4f}")

# Referenz-Vorhersagen (für spätere Verifikation)
ref_predictions = model.predict(x_test[:10], verbose=0)
print(f"Referenz-Vorhersagen (erste 10 Samples): {np.argmax(ref_predictions, axis=1)}")

# ── 2. Format 1: .keras (natives Format) ─────────────────
keras_path = 'saved_model.keras'
model.save(keras_path)
print(f"\nFormat 1: .keras → {keras_path}")
print(f"  Dateigröße: {os.path.getsize(keras_path) / 1024:.1f} KB")

# Format 1 laden
model_keras = keras.models.load_model(keras_path)
_, acc_keras = model_keras.evaluate(x_test, y_test, verbose=0)
preds_keras = model_keras.predict(x_test[:10], verbose=0)
print(f"  Geladene Accuracy: {acc_keras:.4f}")
print(f"  Vorhersagen identisch: {np.allclose(ref_predictions, preds_keras, atol=1e-5)}")

# ── 3. Format 2: SavedModel (TF-Format) ───────────────────
saved_model_path = 'saved_model_tf/'
tf.saved_model.save(model, saved_model_path)
print(f"\nFormat 2: SavedModel → {saved_model_path}")
# Größe des SavedModel-Verzeichnisses
total_size = 0
for dirpath, _, filenames in os.walk(saved_model_path):
    for f in filenames:
        fp = os.path.join(dirpath, f)
        total_size += os.path.getsize(fp)
print(f"  Verzeichnisgröße: {total_size / 1024:.1f} KB")

# Format 2 laden
model_tf = tf.saved_model.load(saved_model_path)
# SavedModel als TF-Funktion aufrufen
infer = model_tf.signatures['serving_default']
x_input = tf.constant(x_test[:10])
output_key = list(infer(x_input).keys())[0]
preds_tf = infer(x_input)[output_key].numpy()
print(f"  Vorhersagen identisch: {np.allclose(ref_predictions, preds_tf, atol=1e-4)}")

# Format 2 alternativ als Keras-Modell laden
model_tf2 = keras.models.load_model(saved_model_path)
_, acc_tf = model_tf2.evaluate(x_test, y_test, verbose=0)
print(f"  Geladene Accuracy: {acc_tf:.4f}")

# ── 4. Format 3: .h5 (Legacy-Format) ─────────────────────
h5_path = 'saved_model.h5'
model.save(h5_path)
print(f"\nFormat 3: .h5 → {h5_path}")
print(f"  Dateigröße: {os.path.getsize(h5_path) / 1024:.1f} KB")

# Format 3 laden
model_h5 = keras.models.load_model(h5_path)
_, acc_h5 = model_h5.evaluate(x_test, y_test, verbose=0)
preds_h5  = model_h5.predict(x_test[:10], verbose=0)
print(f"  Geladene Accuracy: {acc_h5:.4f}")
print(f"  Vorhersagen identisch: {np.allclose(ref_predictions, preds_h5, atol=1e-5)}")

# ── 5. Vergleichstabelle ──────────────────────────────────
print("\n" + "="*60)
print("VERGLEICH DER SPEICHERFORMATE")
print("="*60)

keras_size = os.path.getsize(keras_path) / 1024
h5_size    = os.path.getsize(h5_path) / 1024
tf_size    = total_size / 1024

print(f"{'Format':15} | {'Dateigröße (KB)':15} | {'Accuracy':10} | {'Vorhersagen OK':15}")
print("-" * 60)
print(f"{'keras (.keras)':15} | {keras_size:15.1f} | {acc_keras:.4f}     | "
      f"{str(np.allclose(ref_predictions, preds_keras, atol=1e-5)):15}")
print(f"{'SavedModel':15} | {tf_size:15.1f} | {acc_tf:.4f}     | "
      f"{str(np.allclose(ref_predictions, preds_tf, atol=1e-4)):15}")
print(f"{'HDF5 (.h5)':15} | {h5_size:15.1f} | {acc_h5:.4f}     | "
      f"{str(np.allclose(ref_predictions, preds_h5, atol=1e-5)):15}")

# ── 6. Balkendiagramm Dateigrößen ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Modell Speicherformate: Vergleich', fontsize=13)

ax = axes[0]
formate = ['.keras', 'SavedModel', '.h5']
groessen = [keras_size, tf_size, h5_size]
ax.bar(formate, groessen, color=['blue', 'green', 'orange'], alpha=0.8)
ax.set_title('Dateigröße (KB)')
ax.set_ylabel('Größe (KB)'); ax.grid(True, axis='y')
for i, (f, g) in enumerate(zip(formate, groessen)):
    ax.text(i, g + 0.5, f'{g:.1f} KB', ha='center', fontsize=10)

ax2 = axes[1]
accs = [orig_acc, acc_keras, acc_tf, acc_h5]
labels = ['Original', '.keras', 'SavedModel', '.h5']
ax2.bar(labels, accs, color=['black', 'blue', 'green', 'orange'], alpha=0.8)
ax2.set_title('Accuracy nach Laden')
ax2.set_ylabel('Test-Accuracy')
ax2.set_ylim(0.95, 1.0); ax2.grid(True, axis='y')
for i, (l, a) in enumerate(zip(labels, accs)):
    ax2.text(i, a + 0.001, f'{a:.4f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('A14_1_modell_formate.png', dpi=100)
plt.show()

print("\n✓ Alle 3 Formate erfolgreich gespeichert und geladen!")
print("  Vorhersagen sind numerisch identisch (Floating-Point-Toleranz).")


## Exercise 2

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 14: Modell-Deployment
# Niveau: Anfänger
# Aufgabe 2 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import time
import os

np.random.seed(42)
tf.random.set_seed(42)

print("=== TensorFlow Lite Konvertierung und Quantisierung ===\n")

# ── 1. Daten laden ─────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0
x_train = x_train.reshape(-1, 784)
x_test  = x_test.reshape(-1, 784)

# ── 2. Modell trainieren ──────────────────────────────────
model = keras.Sequential([
    layers.Dense(256, activation='relu', input_shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64,  activation='relu'),
    layers.Dense(10,  activation='softmax')
], name='mnist_for_tflite')

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print("Trainiere Modell...")
model.fit(x_train, y_train, epochs=5, batch_size=256, verbose=1)
_, original_acc = model.evaluate(x_test, y_test, verbose=0)
model.save('model_for_tflite.keras')
original_size = os.path.getsize('model_for_tflite.keras') / 1024
print(f"Originalmodell: Acc={original_acc:.4f}, Größe={original_size:.1f} KB\n")

# ── 3. TFLite Konvertierung (Float32) ─────────────────────
print("Konvertiere zu TFLite (Float32)...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

tflite_path = 'model_float32.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
tflite_size = os.path.getsize(tflite_path) / 1024
print(f"TFLite Float32: {tflite_size:.1f} KB")

# ── 4. TFLite Int8-Quantisierung ──────────────────────────
print("Quantisiere zu Int8...")

# Repräsentative Daten für Kalibrierung
def representative_data_gen():
    for sample in x_train[:200]:
        yield [sample.reshape(1, 784).astype(np.float32)]

converter_quant = tf.lite.TFLiteConverter.from_keras_model(model)
converter_quant.optimizations = [tf.lite.Optimize.DEFAULT]
converter_quant.representative_dataset = representative_data_gen
# Vollständige Int8-Quantisierung (In- und Output bleiben float für Kompatibilität)
converter_quant.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
    tf.lite.OpsSet.TFLITE_BUILTINS
]
converter_quant.inference_input_type  = tf.float32
converter_quant.inference_output_type = tf.float32

tflite_quant_model = converter_quant.convert()
quant_path = 'model_int8.tflite'
with open(quant_path, 'wb') as f:
    f.write(tflite_quant_model)
quant_size = os.path.getsize(quant_path) / 1024
print(f"TFLite Int8:    {quant_size:.1f} KB")

# ── 5. TFLite Inferenz-Funktion ───────────────────────────
def run_tflite_inference(tflite_model_bytes, input_data):
    """Führt Inferenz mit TFLite-Interpreter durch."""
    interpreter = tf.lite.Interpreter(model_content=tflite_model_bytes)
    interpreter.allocate_tensors()

    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    predictions = []
    for sample in input_data:
        interpreter.set_tensor(input_details[0]['index'],
                                 sample.reshape(1, 784).astype(np.float32))
        interpreter.invoke()
        pred = interpreter.get_tensor(output_details[0]['index'])
        predictions.append(pred[0])

    return np.array(predictions)

# ── 6. Accuracy der konvertierten Modelle ─────────────────
print("\nBerechne Accuracy (100 Test-Samples)...")
sample_x = x_test[:100]
sample_y = y_test[:100]

preds_tflite = run_tflite_inference(tflite_model, sample_x)
tflite_acc   = (np.argmax(preds_tflite, axis=1) == sample_y).mean()

preds_quant = run_tflite_inference(tflite_quant_model, sample_x)
quant_acc   = (np.argmax(preds_quant, axis=1) == sample_y).mean()

# Original Keras-Vorhersagen für Vergleich
original_preds = model.predict(sample_x, verbose=0)
orig_acc_100   = (np.argmax(original_preds, axis=1) == sample_y).mean()

print(f"Original Keras:   {orig_acc_100:.4f}")
print(f"TFLite Float32:   {tflite_acc:.4f}")
print(f"TFLite Int8:      {quant_acc:.4f}")

# ── 7. Benchmark: Inferenz-Geschwindigkeit ────────────────
N_BENCHMARK = 100
sample_bench = x_test[:1]   # Einzelner Sample

print(f"\nBenchmark: {N_BENCHMARK} Inferenzen für 1 Sample...")

# Original Keras
t0 = time.time()
for _ in range(N_BENCHMARK):
    _ = model.predict(sample_bench, verbose=0)
keras_time = (time.time() - t0) / N_BENCHMARK * 1000

# TFLite Float32
interp_f32 = tf.lite.Interpreter(model_content=tflite_model)
interp_f32.allocate_tensors()
in_det  = interp_f32.get_input_details()
out_det = interp_f32.get_output_details()
t0 = time.time()
for _ in range(N_BENCHMARK):
    interp_f32.set_tensor(in_det[0]['index'], sample_bench.astype(np.float32))
    interp_f32.invoke()
tflite_time = (time.time() - t0) / N_BENCHMARK * 1000

# TFLite Int8
interp_int8 = tf.lite.Interpreter(model_content=tflite_quant_model)
interp_int8.allocate_tensors()
in_det2  = interp_int8.get_input_details()
out_det2 = interp_int8.get_output_details()
t0 = time.time()
for _ in range(N_BENCHMARK):
    interp_int8.set_tensor(in_det2[0]['index'], sample_bench.astype(np.float32))
    interp_int8.invoke()
quant_time = (time.time() - t0) / N_BENCHMARK * 1000

print(f"Original Keras:  {keras_time:.2f} ms/Sample")
print(f"TFLite Float32: {tflite_time:.2f} ms/Sample")
print(f"TFLite Int8:    {quant_time:.2f} ms/Sample")

# ── 8. Visualisierung ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('TFLite Konvertierung: Vergleich', fontsize=13)

# Größen
ax = axes[0]
modell_typen = ['Original\n(.keras)', 'TFLite\n(Float32)', 'TFLite\n(Int8)']
groessen     = [original_size, tflite_size, quant_size]
bars = ax.bar(modell_typen, groessen, color=['blue', 'green', 'orange'], alpha=0.8)
ax.set_title('Modellgröße (KB)')
ax.set_ylabel('Größe (KB)'); ax.grid(True, axis='y')
for bar, size in zip(bars, groessen):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{size:.1f} KB', ha='center', fontsize=9)

# Accuracy
ax2 = axes[1]
accs = [orig_acc_100, tflite_acc, quant_acc]
ax2.bar(modell_typen, accs, color=['blue', 'green', 'orange'], alpha=0.8)
ax2.set_title('Accuracy (100 Test-Samples)')
ax2.set_ylabel('Accuracy'); ax2.set_ylim(0.9, 1.02); ax2.grid(True, axis='y')
for i, acc in enumerate(accs):
    ax2.text(i, acc + 0.003, f'{acc:.4f}', ha='center', fontsize=9)

# Inferenz-Zeit
ax3 = axes[2]
zeiten = [keras_time, tflite_time, quant_time]
ax3.bar(modell_typen, zeiten, color=['blue', 'green', 'orange'], alpha=0.8)
ax3.set_title('Inferenzzeit (ms/Sample)')
ax3.set_ylabel('Zeit (ms)'); ax3.grid(True, axis='y')
for i, zeit in enumerate(zeiten):
    ax3.text(i, zeit + 0.02, f'{zeit:.2f}ms', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('A14_2_tflite_vergleich.png', dpi=100)
plt.show()

print(f"\nGrößenreduktion: {((original_size - quant_size)/original_size*100):.1f}% kleiner (Int8)")
print(f"Accuracy-Verlust (Int8): {(orig_acc_100 - quant_acc)*100:.2f} Prozentpunkte")


## Exercise 3

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 14: Modell-Deployment
# Niveau: Anfänger
# Aufgabe 3 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import json
import time

np.random.seed(42)
tf.random.set_seed(42)

print("=== REST API Simulation für MNIST-Modell ===\n")

# ── 1. Modell trainieren ──────────────────────────────────
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0
x_train = x_train.reshape(-1, 784)
x_test  = x_test.reshape(-1, 784)

model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(784,)),
    layers.Dense(64,  activation='relu'),
    layers.Dense(10,  activation='softmax')
], name='mnist_api_model')
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=5, batch_size=256, verbose=1)
_, acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nModell Test-Accuracy: {acc:.4f}")

# ── 2. Validierungs-Funktion ──────────────────────────────
class ValidationError(Exception):
    pass

def validate_input(input_json):
    """
    Validiert den API-Eingang.
    Erwartet: {"pixels": [784 Werte zwischen 0 und 255]}
    """
    # Pflichtfelder prüfen
    if 'pixels' not in input_json:
        raise ValidationError("Fehler: 'pixels' Feld fehlt in der Anfrage")

    pixels = input_json['pixels']

    # Typ prüfen
    if not isinstance(pixels, list):
        raise ValidationError(f"Fehler: 'pixels' muss eine Liste sein, nicht {type(pixels)}")

    # Länge prüfen
    if len(pixels) != 784:
        raise ValidationError(f"Fehler: 'pixels' muss genau 784 Werte enthalten, erhalten: {len(pixels)}")

    # Wertebereich prüfen
    pixels_arr = np.array(pixels, dtype=float)
    if np.any(pixels_arr < 0) or np.any(pixels_arr > 255):
        raise ValidationError("Fehler: Pixelwerte müssen im Bereich [0, 255] liegen")

    return pixels_arr

def preprocess_input(pixels_arr):
    """
    Vorverarbeitung: Normalisierung auf [0, 1].
    """
    normalized = pixels_arr / 255.0
    return normalized.reshape(1, 784).astype(np.float32)

# ── 3. Predict API (simuliert Flask/FastAPI Endpunkt) ─────
def predict_api(input_json):
    """
    Simuliert POST /predict REST-Endpunkt.

    Erwartet:
        input_json: {"pixels": [784 float Werte im Bereich 0-255]}

    Gibt zurück:
        JSON-Response mit Vorhersage und Konfidenz
    """
    request_time = time.time()
    response = {
        'status':     None,
        'prediction': None,
        'confidence': None,
        'probabilities': None,
        'error':      None,
        'processing_time_ms': None
    }

    try:
        # 1. Validierung
        pixels_arr = validate_input(input_json)

        # 2. Vorverarbeitung
        model_input = preprocess_input(pixels_arr)

        # 3. Inferenz
        probabilities = model.predict(model_input, verbose=0)[0]
        predicted_class = int(np.argmax(probabilities))
        confidence      = float(probabilities[predicted_class])

        # 4. Antwort aufbauen
        response['status']     = 'success'
        response['prediction'] = predicted_class
        response['confidence'] = round(confidence, 4)
        response['probabilities'] = {
            str(i): round(float(p), 4) for i, p in enumerate(probabilities)
        }

    except ValidationError as ve:
        response['status'] = 'validation_error'
        response['error']  = str(ve)

    except Exception as e:
        response['status'] = 'server_error'
        response['error']  = f"Interner Fehler: {str(e)}"

    finally:
        response['processing_time_ms'] = round((time.time() - request_time) * 1000, 2)

    return response

# ── 4. API mit 5 Beispiel-Inputs testen ───────────────────
print("\n" + "="*60)
print("API TEST – 5 Beispiel-Anfragen")
print("="*60)

# Valide Anfragen
test_inputs = [
    # Echte MNIST-Bilder
    {'pixels': (x_test[0] * 255).tolist(), 'true_label': int(y_test[0])},
    {'pixels': (x_test[1] * 255).tolist(), 'true_label': int(y_test[1])},
    {'pixels': (x_test[42] * 255).tolist(), 'true_label': int(y_test[42])},
    # Graubild (alle Pixel = 128 → schwierig)
    {'pixels': [128.0] * 784, 'true_label': None},
    # Weißes Bild (alle Pixel = 0)
    {'pixels': [0.0] * 784, 'true_label': None},
]

for i, test in enumerate(test_inputs):
    request = {'pixels': test['pixels']}
    response = predict_api(request)
    true_lbl = test.get('true_label', '?')
    print(f"\n--- Anfrage {i+1} ---")
    print(f"  Status:     {response['status']}")
    if response['status'] == 'success':
        print(f"  Vorhersage: {response['prediction']} (wahr: {true_lbl})")
        print(f"  Konfidenz:  {response['confidence']:.4f}")
        correct = response['prediction'] == true_lbl if true_lbl is not None else 'N/A'
        print(f"  Korrekt:    {correct}")
    else:
        print(f"  Fehler:     {response['error']}")
    print(f"  Zeit:       {response['processing_time_ms']} ms")

# ── 5. Fehlerbehandlung testen ────────────────────────────
print("\n" + "="*60)
print("FEHLERBEHANDLUNG – Ungültige Anfragen")
print("="*60)

invalid_inputs = [
    {'name': 'Fehlendes Feld', 'data': {'bild': [0.5] * 784}},
    {'name': 'Falsche Länge',  'data': {'pixels': [0.5] * 100}},
    {'name': 'Falscher Typ',   'data': {'pixels': 'ungültige Zeichenkette'}},
    {'name': 'Außerhalb [0,255]', 'data': {'pixels': [-10.0] + [128.0] * 783}},
]

for inv in invalid_inputs:
    response = predict_api(inv['data'])
    print(f"\n  Test: {inv['name']}")
    print(f"  Status: {response['status']}")
    print(f"  Fehler: {response['error']}")

# ── 6. Konfidenz-Visualisierung ───────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle('API Vorhersagen auf MNIST-Testsamples', fontsize=13)

for i in range(5):
    img_flat = x_test[i]
    request  = {'pixels': (img_flat * 255).tolist()}
    response = predict_api(request)

    axes[i].imshow(img_flat.reshape(28, 28), cmap='gray')
    pred = response['prediction']
    conf = response['confidence']
    true = y_test[i]
    color = 'green' if pred == true else 'red'
    axes[i].set_title(f'Pred: {pred} (Conf: {conf:.0%})\nWahr: {true}',
                       color=color, fontsize=9)
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('A14_3_api_vorhersagen.png', dpi=100)
plt.show()

print("\nREST API Simulation abgeschlossen!")
print("In der Produktion wird predict_api() als Flask/FastAPI-Endpunkt eingebunden.")
